# Olist: dự đoán đơn sẽ bị review xấu, so sánh các model kinh điển

**Bài toán.** Ngay khi một đơn được giao tới khách, đội CSKH muốn biết đơn nào có nguy cơ nhận review
1–2 sao để liên hệ chủ động. Nhãn `bad_review = review_score <= 2`, mỗi đơn giữ review có
`review_answer_timestamp` mới nhất. Feature chỉ dùng thông tin **đã biết lúc giao hàng**: thời gian giao,
trễ bao nhiêu so với ngày hẹn, số món, giá, tỉ lệ phí ship, số kỳ trả góp. Thời điểm dự đoán là **lúc đơn vừa
giao**, giống module 10: feature giao hàng hợp lệ, còn mọi cột review (điểm, bình luận, ngày tạo, ngày trả
lời) là leakage.

**Chia theo thời gian** (định nghĩa chung của track): train là đơn mua trước 2018-01-01, validation từ
2018-01-01 tới trước 2018-04-01, test từ 2018-04-01 tới hết 2018-08-31. Cách dựng feature an toàn với
`Pipeline` và CV theo thời gian là chủ đề của module 10; ở đây tập trung vào model và metric.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore`) đọc dữ liệu thật, train model, vẽ. Test bỏ qua chúng.
- Chấm: `uv run pytest tracks/data-science/modules/11-classic-ml/exercises/intermediate/01-olist-bad-review-models -q`

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score

FEATURES = ["delivery_days", "late_days", "estimated_days", "n_items", "n_sellers",
            "total_price", "freight_ratio", "max_installments"]

## 1. Dựng bảng dữ liệu

Một dòng mỗi đơn **giao xong** (`order_status == "delivered"` và có `order_delivered_customer_date`),
mua trước 2018-09-01, có review và có ít nhất một item. Feature (đơn vị ngày là số thực, ví dụ 5.5):

| Cột | Cách tính |
| --- | --- |
| `delivery_days` | giao thực tế − thời điểm mua |
| `late_days` | giao thực tế − ngày hẹn (âm là giao sớm) |
| `estimated_days` | ngày hẹn − thời điểm mua |
| `n_items`, `n_sellers` | số dòng item, số seller khác nhau |
| `total_price` | tổng `price` |
| `freight_ratio` | tổng `freight_value` / (tổng `price` + tổng `freight_value`) |
| `max_installments` | `payment_installments` lớn nhất; đơn không có payment coi là 1 |

In [ ]:
def build_dataset(orders: pd.DataFrame, reviews: pd.DataFrame, items: pd.DataFrame,
                  payments: pd.DataFrame) -> pd.DataFrame:
    """Trả DataFrame cột order_id, order_purchase_timestamp, FEATURES..., bad_review (0/1)."""
    d = orders[(orders.order_status == "delivered") & orders.order_delivered_customer_date.notna()]
    d = d[d.order_purchase_timestamp < "2018-09-01"]
    latest = reviews.sort_values("review_answer_timestamp").drop_duplicates("order_id", keep="last")
    df = d.merge(latest[["order_id", "review_score"]], on="order_id", how="inner")
    it = items.groupby("order_id").agg(
        n_items=("order_item_id", "size"), n_sellers=("seller_id", "nunique"),
        total_price=("price", "sum"), total_freight=("freight_value", "sum"))
    pay = payments.groupby("order_id").agg(max_installments=("payment_installments", "max"))
    df = df.merge(it, on="order_id", how="inner").merge(pay, on="order_id", how="left")
    day = pd.Timedelta(days=1)
    df["delivery_days"] = (df.order_delivered_customer_date - df.order_purchase_timestamp) / day
    df["late_days"] = (df.order_delivered_customer_date - df.order_estimated_delivery_date) / day
    df["estimated_days"] = (df.order_estimated_delivery_date - df.order_purchase_timestamp) / day
    df["freight_ratio"] = df.total_freight / (df.total_price + df.total_freight)
    df["max_installments"] = df.max_installments.fillna(1)
    df["bad_review"] = (df.review_score <= 2).astype(int)
    return df[["order_id", "order_purchase_timestamp", *FEATURES, "bad_review"]].reset_index(drop=True)


def time_split(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """(train, val, test) theo order_purchase_timestamp, mốc 2018-01-01, 2018-04-01, 2018-09-01."""
    t = df.order_purchase_timestamp
    train = df[t < "2018-01-01"]
    val = df[(t >= "2018-01-01") & (t < "2018-04-01")]
    test = df[(t >= "2018-04-01") & (t < "2018-09-01")]
    return train, val, test

In [ ]:
# explore: đọc Olist thật (tải bằng: uv run python data/download_olist.py)
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW = ROOT / "data" / "raw"
dates = ["order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"]
orders = pd.read_csv(RAW / "olist_orders_dataset.csv", parse_dates=dates)
reviews = pd.read_csv(RAW / "olist_order_reviews_dataset.csv", parse_dates=["review_answer_timestamp"])
items = pd.read_csv(RAW / "olist_order_items_dataset.csv")
payments = pd.read_csv(RAW / "olist_order_payments_dataset.csv")
df = build_dataset(orders, reviews, items, payments)
train, val, test = time_split(df)
print(len(df), round(df.bad_review.mean(), 4))

## 2. Ba tập có giống nhau không?

Trước khi train bất cứ gì: so tỉ lệ review xấu và tỉ lệ giao trễ giữa train, validation, test. Nếu khác
nhiều, điều đó ảnh hưởng tới metric nào (ROC-AUC? PR-AUC? Brier?) và tới việc chọn ngưỡng ra sao?

In [ ]:
# explore: phân phối nhãn và tỉ lệ trễ theo tập
pd.DataFrame({
    name: {"n": len(s), "bad_rate": s.bad_review.mean(), "late_rate": (s.late_days > 0).mean(),
           "bad_rate_if_late": s.loc[s.late_days > 0, "bad_review"].mean()}
    for name, s in [("train", train), ("val", val), ("test", test)]
}).T.round(3)

## 3. Metric và bảng so sánh model

`evaluate(y, p)` trả bốn metric: `roc_auc`, `pr_auc` (average precision), `log_loss`, `brier`. Được dùng
`sklearn.metrics` (bài Basic 01 đã tự cài).

`compare_models(models, train, val)`: fit từng model trên `train[FEATURES]`, dự đoán xác suất trên `val`.
Trả DataFrame index là tên model, cột là 4 metric trên validation cộng `train_roc_auc` (ROC-AUC trên chính
tập train, để thấy overfitting), sắp theo `pr_auc` giảm dần.

In [ ]:
def evaluate(y, p) -> dict[str, float]:
    p = np.clip(np.asarray(p, dtype=float), 1e-15, 1 - 1e-15)
    return {
        "roc_auc": roc_auc_score(y, p),
        "pr_auc": average_precision_score(y, p),
        "log_loss": log_loss(y, p, labels=[0, 1]),
        "brier": brier_score_loss(y, p),
    }


def compare_models(models: dict, train: pd.DataFrame, val: pd.DataFrame) -> pd.DataFrame:
    rows = {}
    for name, model in models.items():
        model.fit(train[FEATURES], train.bad_review)
        row = evaluate(val.bad_review, model.predict_proba(val[FEATURES])[:, 1])
        row["train_roc_auc"] = roc_auc_score(train.bad_review, model.predict_proba(train[FEATURES])[:, 1])
        rows[name] = row
    return pd.DataFrame(rows).T.sort_values("pr_auc", ascending=False)

In [ ]:
# explore: năm họ model kinh điển. kNN và logistic cần scale; cây thì không.
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

models = {
    "logistic": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "knn_50": make_pipeline(StandardScaler(), KNeighborsClassifier(50)),
    "tree_depth5": DecisionTreeClassifier(max_depth=5, random_state=0),
    "tree_full": DecisionTreeClassifier(random_state=0),
    "random_forest": RandomForestClassifier(300, min_samples_leaf=20, n_jobs=-1, random_state=0),
}
table = compare_models(models, train, val)
table.round(3)

Đọc bảng: model nào overfit (khoảng cách `train_roc_auc` với `roc_auc`)? Vì sao `log_loss` của cây đầy đủ
tệ hơn hẳn các model khác dù ROC-AUC chỉ kém vừa phải? PR-AUC của một model đoán bừa trên validation là
bao nhiêu?

SVM (`SVC`) không có trong bảng: với khoảng 43 nghìn dòng, kernel RBF tốn thời gian bậc hai theo số dòng,
và từ sklearn 1.9 `probability=True` đã deprecated (dùng `CalibratedClassifierCV`). Thử trên mẫu 5 000
dòng nếu muốn.

## 4. Validation có đại diện cho tương lai không?

Chấm model tốt nhất trên **test**. So với validation, ROC-AUC và PR-AUC thay đổi thế nào? Dùng bảng ở mục
2 để giải thích.

In [ ]:
# explore: chấm random forest trên test
best = models["random_forest"]
p_val = best.predict_proba(val[FEATURES])[:, 1]
p_test = best.predict_proba(test[FEATURES])[:, 1]
pd.DataFrame({"val": evaluate(val.bad_review, p_val), "test": evaluate(test.bad_review, p_test)}).round(3)

## 5. Chọn ngưỡng theo chi phí

Giả định: liên hệ một đơn tốn 5 R$ (FP), bỏ lỡ một đơn sẽ bị review xấu tốn 30 R$ (FN). Chọn ngưỡng trên
**validation**, rồi đo chi phí trên **test** và so với ngưỡng 0.5 và ngưỡng lý thuyết 5/35 (bài Basic 02).

In [ ]:
# explore: quét ngưỡng trên validation, đo trên test
def cost_per_order(y, p, t, c_fp=5.0, c_fn=30.0):
    y = np.asarray(y).astype(bool)
    pred = p >= t
    return (c_fp * np.sum(~y & pred) + c_fn * np.sum(y & ~pred)) / len(y)

grid = np.round(np.arange(0.02, 0.9, 0.005), 3)
t_val = grid[np.argmin([cost_per_order(val.bad_review, p_val, t) for t in grid])]
for label, t in [("0.5", 0.5), ("lý thuyết 5/35", 5 / 35), ("chọn trên val", t_val)]:
    print(f"{label:>15}: t={t:.3f}  chi phí/đơn trên test={cost_per_order(test.bad_review, p_test, t):.3f}  "
          f"tỉ lệ liên hệ={np.mean(p_test >= t):.3f}")
print("không liên hệ ai:", round(cost_per_order(test.bad_review, p_test, np.inf), 3))

## 6. Câu hỏi để tự trả lời

1. Nếu chỉ được báo cáo một metric cho đội CSKH, bạn chọn metric nào và vì sao?
2. Tỉ lệ review xấu ở validation (18%) khác train (12%) và test (10%). Ngưỡng chọn trên validation có còn
   tối ưu ở test không? Bạn sẽ làm gì khi đưa model lên production?
3. Feature `late_days` chỉ biết sau khi giao. Nếu đội CSKH muốn can thiệp **trước** khi giao, đó là bài toán
   với thời điểm dự đoán khác (lúc mua hoặc lúc giao cho hãng vận chuyển). Những feature nào phải bỏ, và bạn
   kỳ vọng ROC-AUC thay đổi thế nào?